In [4]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

data_dir = Path("/Volumes/tytelllab$/NewData/ElongateFoils/ElongateFoil_DorsalandAnalFins/RawData/ForceData/2026-09-30/nofin")          # folder containing the .h5 files
files = [data_dir / f"test{i:03d}.h5" for i in range(1, 28)]   # test001 ... test027


def analyze_file(path):
    with h5py.File(path, "r") as f:
        t     = f["NominalStimulus/t"][:]
        theta = np.deg2rad(f["NominalStimulus/Position"][:])
        Fx    = f["Calibrated/xForce"][:]
        Fy    = f["Calibrated/yForce"][:]

        stim = f["NominalStimulus"].attrs
        meta = {
            "amplitude_deg": stim["Amplitude"],
            "frequency_hz":  stim["Frequency"],
            "cycles":        stim["Cycles"],
            "foil_type":     f.attrs.get("Foiltype", ""),
            "flow_speed_mm": f.attrs.get("FlowSpeed_mm", np.nan),
        }

    # Remove static offset using the pre-motion period
    pre = t < 0
    Fx = Fx - Fx[pre].mean()
    Fy = Fy - Fy[pre].mean()

    # Rotate into the fixed frame
    F_stream  = Fx*np.cos(theta) - Fy*np.sin(theta)
    F_lateral = Fx*np.sin(theta) + Fy*np.cos(theta)
    thrust = F_stream

    # Use only whole cycles of motion
    duration = meta["cycles"] / meta["frequency_hz"]
    moving = (t >= 0) & (t < duration)

    results = {
        "file": path.name,
        **meta,
        "mean_thrust":      thrust[moving].mean(),
        "peak_thrust":      thrust[moving].max(),
        "mean_lateral":     F_lateral[moving].mean(),
        "peak_lateral":     np.abs(F_lateral[moving]).max(),
    }
    return results, (t, theta, thrust, F_lateral)


rows = []
traces = {}

for path in files:
    if not path.exists():
        print(f"Skipping {path.name}: not found")
        continue
    try:
        res, trace = analyze_file(path)
        rows.append(res)
        traces[path.name] = trace
        print(f"{path.name}: mean thrust = {res['mean_thrust']:.4f}, "
              f"peak lateral = {res['peak_lateral']:.4f}")
    except Exception as e:
        print(f"Skipping {path.name}: {e}")

df = pd.DataFrame(rows)
df.to_csv(data_dir / "thrust_lateral_summary.csv", index=False)
print(df)



test001.h5: mean thrust = -0.0041, peak lateral = 0.0721
test002.h5: mean thrust = -0.0029, peak lateral = 0.0730
test003.h5: mean thrust = -0.0028, peak lateral = 0.0763
test004.h5: mean thrust = -0.0067, peak lateral = 0.1270
test005.h5: mean thrust = -0.0101, peak lateral = 0.1190
test006.h5: mean thrust = -0.0077, peak lateral = 0.1212
test007.h5: mean thrust = -0.0042, peak lateral = 0.5188
test008.h5: mean thrust = -0.0036, peak lateral = 0.5119
test009.h5: mean thrust = -0.0044, peak lateral = 0.4936
test010.h5: mean thrust = -0.0022, peak lateral = 0.0472
test011.h5: mean thrust = -0.0018, peak lateral = 0.0491
test012.h5: mean thrust = -0.0023, peak lateral = 0.0477
test013.h5: mean thrust = -0.0058, peak lateral = 0.0720
test014.h5: mean thrust = -0.0050, peak lateral = 0.0751
test015.h5: mean thrust = -0.0080, peak lateral = 0.0649
test016.h5: mean thrust = -0.0077, peak lateral = 0.1555
test017.h5: mean thrust = -0.0097, peak lateral = 0.1461
test018.h5: mean thrust = -0.00